In [1]:
import numpy as np
import pandas as pd

import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Carnet_de_demarrage.ipynb
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/Enonce.md
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/a_decider.csv
/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/train.csv


In [2]:
import pandas as pd

DOSSIER = "/kaggle/input/datasets/yannahomlanto/esgc-vak-sujet-07/Sujet_07_Santander_Transaction-20261110T150120Z-1-001/Sujet_07_Santander_Transaction/"

train = pd.read_csv(DOSSIER + "train.csv")
a_decider = pd.read_csv(DOSSIER + "a_decider.csv")

print("train     :", train.shape)
print("a_decider :", a_decider.shape)

train     : (160000, 202)
a_decider : (40000, 201)


> **Usage de l'IA.** Les outils de suivi et les mesures de 1.1 et 1.8 ont été écrits avec l'aide de Claude (Anthropic), puis relus, exécutés et vérifiés par l'équipe.

## 0 — Outils de suivi

Copie conforme de `notebook_Outils_de_suivi.ipynb`. Chaque outil remplit un tableau qu'on rend :

| Outil | La question |
|---|---|
| `prevoir` / `noter` / `essai` | qu'ai-je annoncé, et l'ai-je mesuré après ? |
| `constat` | qu'ai-je vu dans les données, avec quel chiffre ? |
| `hypothese` | qu'est-ce que je crois, et qu'est-ce qui me contredirait ? |
| `journal` | qu'ai-je décidé, et pourquoi, sans parler du score ? |

Seul changement : `barre` (la barre de bruit) sera mesurée à l'étape 2.2. En attendant elle vaut `None`, et `journal` l'indique au lieu de planter.

In [3]:
# 0 — outils de suivi (copie conforme de notebook_Outils_de_suivi)
from datetime import datetime
from IPython.display import display

ID = "ID_code"     # la colonne identifiant
CIBLE = "target"   # la colonne à prédire

# la barre de bruit sera mesurée à l'étape 2.2 : None en attendant
barre = None

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p['essai'] != essai]
    PREDICTIONS.append({'essai': essai, 'attendu': attendu, 'raison': raison,
                        'écrit à': datetime.now().strftime('%H:%M:%S')})
    display(pd.DataFrame(PREDICTIONS))

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. Refuse un essai non annoncé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == hypothese]
    if not prevu:
        print(f'  refusé : aucune prédiction pour « {hypothese} ».')
        print('  Écrivez prevoir(...) d\'abord — l\'ordre est vérifié, pas seulement demandé.')
        return
    ligne = {'hypothèse': hypothese, 'changement': changement, 'attendu': attendu,
             'mesuré': round(score, 2),
             'prédiction écrite avant ?': 'oui, à ' + prevu[0]['écrit à']}
    ESSAIS[:] = [e for e in ESSAIS if e['hypothèse'] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

def essai(nom, mesure, changement='', attendu=''):
    """Le chemin strict : la mesure est prise ici, donc l'ordre est prouvé."""
    prevu = [p for p in PREDICTIONS if p['essai'] == nom]
    if not prevu:
        print(f'refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d\'abord.')
        return None
    score = 100 * mesure()
    noter(nom, changement or 'voir la question', attendu or prevu[0]['attendu'], score)
    return score

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? · conséquence : garder, écarter, surveiller."""
    ligne = {'observation': observation, 'chiffre': chiffre,
             'comparaison': comparaison, 'conséquence': consequence}
    CONSTATS[:] = [c for c in CONSTATS if c['observation'] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT."""
    ligne = {'feature': feature, 'ce qu\'elle raconte': ce_qu_elle_raconte,
             'test': comment_on_la_test, 'contredite si': ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h['feature'] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

JOURNAL = []

def journal(decision, choix, raison, effet):
    """Décision, choix, raison d'audit, effet mesuré — comparé au double de la barre."""
    JOURNAL.append({'décision': decision, 'choix': choix,
                    'raison (sans le score)': raison,
                    'effet mesuré': round(effet, 2),
                    # SEUL CHANGEMENT : tant que la barre n'est pas mesurée (2.2), on ne compare pas
                    '> double de la barre ?': ('oui' if abs(effet) > 2 * barre else 'non')
                                              if barre is not None else 'barre pas encore mesurée'})
    display(pd.DataFrame(JOURNAL))

## Étape 1.1 — Inventaire

On a reçu 4 fichiers, dont 2 tables de données. Les 2 autres sont l'énoncé (Enonce.md) et le carnet de démarrage (Carnet_de_demarrage.ipynb).

| Table | Lignes | Colonnes |
|---|---|---|
| train.csv | 160 000 | 202 |
| a_decider.csv | 40 000 | 201 |

a_decider a une colonne de moins : il lui manque `target`. Elle contient les clients sur lesquels on doit prédire ; si on avait leur target, il n'y aurait rien à prédire.

Les chiffres correspondent à ceux de l'énoncé, donc rien n'est tronqué.

In [4]:
print(set(train.columns) - set(a_decider.columns))

X = train.drop(columns=["ID_code", "target"])
print(X.shape)

{'target'}
(160000, 200)


In [5]:
# le constat de l'étape 1.1 : les dimensions comparées à celles de l'énoncé
conforme = train.shape == (160_000, 202) and a_decider.shape == (40_000, 201)
constat("les tables ont la taille annoncée",
        f"train {train.shape[0]} × {train.shape[1]}, a_decider {a_decider.shape[0]} × {a_decider.shape[1]}",
        "énoncé : 160 000 × 202 et 40 000 × 201",
        "conforme : rien n'est tronqué" if conforme else "ÉCART : retélécharger les données")

,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué


Le `constat` range ce résultat dans le tableau des constats : l'observation, le chiffre, la comparaison avec l'énoncé et la conséquence.

**À savoir dire sans carnet.** On compare les dimensions lues à celles de l'énoncé, parce qu'un fichier tronqué au téléchargement donne des scores faux sans aucune erreur. Si on oubliait la colonne `target` dans la comparaison (201 au lieu de 202 pour `train`), le constat crierait à tort.

## Étape 1.8 — Informations indisponibles au moment de la décision

Question posée à chaque colonne : le jour où la banque décide, cette valeur existe-t-elle déjà ?

| Colonne | Décision | Raison |
|---|---|---|
| `target` | mise à part (c'est la cible `y`) | c'est ce qu'on veut prédire : la banque ne la connaît pas au moment de décider. La laisser dans les colonnes du modèle, ce serait lui donner la réponse. |
| `ID_code` | écartée | c'est un identifiant : il désigne le client, il ne le décrit pas. Le modèle apprendrait des numéros par cœur, inutiles sur de nouveaux clients. |
| `var_0` … `var_199` | gardées | elles sont toutes présentes dans `a_decider.csv`, la table des clients sur qui on décide : elles existent donc au moment de la décision. |

Vérification : la seule colonne de `train` absente de `a_decider` est `target`. Après retrait, la table du modèle `X` fait 160 000 lignes et 200 colonnes.

Aucune valeur ne manque (voir étape 1.6) : le signe de fuite « colonne vide quand la réponse n'est pas connue » ne peut pas apparaître ici.

Comme les variables sont anonymes, on ne peut pas se contenter de raisonner : on **mesure** aussi les deux signes de fuite vus en J2 §5, plus l'identifiant.

- **le vide** : une colonne remplie seulement quand l'issue est connue ;
- **la valeur** : une variable qui, seule, sépare presque parfaitement les deux classes (AUC > 0,90) ;
- **l'identifiant** : unique, et sans lien avec la cible.

In [6]:
from sklearn.metrics import roc_auc_score

y = train[CIBLE]

# signe du vide : combien de trous dans les deux tables ?
trous = int(train.isna().sum().sum() + a_decider.isna().sum().sum())
constat("signe du vide : aucune colonne n'a de trou", f"{trous} trou",
        f"sur {train.size + a_decider.size} cellules (train + a_decider)",
        "aucune colonne ne peut trahir la cible par son vide")

# signe de la valeur : l'AUC de chaque variable prise seule
auc = pd.Series({col: roc_auc_score(y, X[col]) for col in X.columns})
auc = np.maximum(auc, 1 - auc)   # une variable qui classe « à l'envers » sépare aussi
display(auc.sort_values(ascending=False).head(5).round(4).to_frame("AUC seule"))
constat("signe de la valeur : aucune variable seule ne connaît la réponse",
        f"AUC seule maximale {auc.max():.4f} ({auc.idxmax()})",
        f"seuil d'alarme 0,90 : {(auc > 0.90).sum()} variable au-dessus",
        "les 200 variables sont gardées")

# l'identifiant : unique ? son numéro dit-il quelque chose de la cible ?
numero = train[ID].str.extract(r"(\d+)$")[0].astype(int)   # "train_123" -> 123
auc_numero = roc_auc_score(y, numero)
communs = len(set(train[ID]) & set(a_decider[ID]))
constat("l'identifiant désigne une ligne, il ne décrit pas le client",
        f"{train[ID].nunique()} valeurs pour {len(train)} lignes ; AUC du numéro {max(auc_numero, 1 - auc_numero):.4f}",
        f"0,5 = hasard ; {communs} ID en commun avec a_decider",
        "l'écarter")

# la liste des colonnes écartées, avec leur raison
journal(ID, "écarté", "il désigne le client au lieu de le décrire : il ne se reproduit pas chez un nouveau client", 0.00)

,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...


,AUC seule
var_81,0.5681
var_139,0.5668
var_6,0.5597
var_12,0.5569
var_110,0.5569


,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...
2,signe de la valeur : aucune variable seule ne ...,AUC seule maximale 0.5681 (var_81),"seuil d'alarme 0,90 : 0 variable au-dessus",les 200 variables sont gardées


,observation,chiffre,comparaison,conséquence
0,les tables ont la taille annoncée,"train 160000 × 202, a_decider 40000 × 201",énoncé : 160 000 × 202 et 40 000 × 201,conforme : rien n'est tronqué
1,signe du vide : aucune colonne n'a de trou,0 trou,sur 40360000 cellules (train + a_decider),aucune colonne ne peut trahir la cible par son...
2,signe de la valeur : aucune variable seule ne ...,AUC seule maximale 0.5681 (var_81),"seuil d'alarme 0,90 : 0 variable au-dessus",les 200 variables sont gardées
3,"l'identifiant désigne une ligne, il ne décrit ...",160000 valeurs pour 160000 lignes ; AUC du num...,"0,5 = hasard ; 0 ID en commun avec a_decider",l'écarter


,décision,choix,raison (sans le score),effet mesuré,> double de la barre ?
0,ID_code,écarté,il désigne le client au lieu de le décrire : i...,0.0,barre pas encore mesurée


| Ce qu'on vérifie | Mesure | Conclusion |
|---|---|---|
| Signe du vide | 0 trou dans `train`, 0 dans `a_decider` | aucune colonne ne peut trahir la cible par son vide |
| Signe de la valeur | meilleure variable seule : `var_81`, AUC 0,5681 ; 0 variable au-dessus de 0,90 | aucune variable ne « connaît » la réponse : les 200 sont gardées |
| `ID_code` unique ? | 160 000 valeurs pour 160 000 lignes | il désigne une ligne, il ne la décrit pas |
| `ID_code` lié à la cible ? | AUC de son numéro : 0,502 | pur hasard : le numéro ne dit rien du client |
| Client présent des deux côtés ? | 0 `ID_code` en commun avec `a_decider` | aucun client n'est à la fois appris et à décider |

**Colonnes écartées : `ID_code` seul.** Ce n'est pas une supposition, c'est la mesure qui tranche.

**À savoir dire sans carnet.** On calcule l'AUC de chaque variable seule et on garde `max(auc, 1 − auc)`, parce qu'une variable qui classe à l'envers sépare aussi bien. Le seuil de 0,90 sert d'alarme : une variable aussi forte à elle seule porterait probablement la réponse. Sans le `max`, une variable avec une AUC de 0,02 (parfaite, mais à l'envers) passerait inaperçue.